In [1]:
!python -m pip install pandas corus nltk scikit-learn seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 4.1 MB/s eta 0:00:00


Были проблемы с библиотеками которые завязаны на curos, поэтому просто через curl скачиваем

In [2]:
!curl -L -O https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  502M  100  502M    0     0   102M      0  0:00:04  0:00:04 --:--:--  123M


In [3]:
!pip install nltk
import nltk

In [4]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
import re
import os
import urllib.request

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import seaborn as sns
import numpy as np
from corus import load_lenta

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

In [6]:
import random

# Фиксируем воспроизводимость результатов

In [7]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [8]:
base_dir = 'lenta-ru-news.csv.gz'
records = load_lenta(base_dir)

data =[]

# берем топики нужные
for record in records:
    data.append({
        'title': record.title,
        'text': record.text,
        'topic': record.topic
    })

df = pd.DataFrame(data)

In [9]:
df.head()

,title,text,topic
0,Названы регионы России с самой высокой смертно...,Вице-премьер по социальным вопросам Татьяна Го...,Россия
1,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт
2,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия
3,В США раскрыли сумму расходов на расследование...,С начала расследования российского вмешательст...,Мир
4,Хакеры рассказали о планах Великобритании зами...,Хакерская группировка Anonymous опубликовала н...,Мир


In [10]:
!pip install pymorphy3
import pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 63.4 MB/s eta 0:00:00


In [11]:
from pymorphy3 import MorphAnalyzer

In [12]:
ru_stopwords = set(stopwords.words('russian'))

In [13]:
stemmer = SnowballStemmer(language='russian')

In [14]:
morph = MorphAnalyzer()

re_url = re.compile(r'http\S+|www\S+')
re_non_alpha = re.compile(r'[^а-яА-ЯёЁ\s]')

def preprocess_text(text):

    if not isinstance(text, str):
        return ""
    text = text.lower()

    text = re_url.sub('', text)

    text = re_non_alpha.sub(' ', text)

    tokens = []
    for word in text.split():
        if word and word not in ru_stopwords:
            # Стемминг слова, пробовал лемматизацию но там не дождался обработки
            stem = stemmer.stem(word)
            tokens.append(stem)

    return ' '.join(tokens)

In [15]:
# Объединим заголовок и текст для получения более полного контекста

df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

In [16]:
df.head()

,title,text,topic,content
0,Названы регионы России с самой высокой смертно...,Вице-премьер по социальным вопросам Татьяна Го...,Россия,Названы регионы России с самой высокой смертно...
1,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт,Австрия не представила доказательств вины росс...
2,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия,Обнаружено самое счастливое место на планете С...
3,В США раскрыли сумму расходов на расследование...,С начала расследования российского вмешательст...,Мир,В США раскрыли сумму расходов на расследование...
4,Хакеры рассказали о планах Великобритании зами...,Хакерская группировка Anonymous опубликовала н...,Мир,Хакеры рассказали о планах Великобритании зами...


In [20]:
df_clean, t = train_test_split(
    df,
    train_size=100000,
    random_state=RANDOM_STATE,
    stratify=df['topic']

)

df_clean = df_clean.reset_index(drop=True)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

окей чета какая то ошибка, исправляем ниже

In [17]:
counts = df['topic'].value_counts()
valid_topics = counts[counts >= 50].index

df_clean = df[df['topic'].isin(valid_topics)]

In [18]:
df_final, t = train_test_split(
    df_clean,
    train_size=100000,
    random_state=RANDOM_STATE,
    stratify=df_clean['topic']

)

df_final = df_final.reset_index(drop=True)

In [19]:
df_final.head()

,title,text,topic,content
0,Государственный департамент США не устраивают ...,"Государственный департамент США заявил, что за...",Мир,Государственный департамент США не устраивают ...
1,Южная Осетия будет просить Россию отменить эко...,Власти Южной Осетии намерены просить Москву от...,Бывший СССР,Южная Осетия будет просить Россию отменить эко...
2,На выходных в Москве прошла пушкинская Велоночь,С 27 на 28 сентября состоялась Восьмая Московс...,Дом,На выходных в Москве прошла пушкинская Велоноч...
3,В Лондоне при взрыве пострадал рабочий,"В центре Лондона 7 февраля произошел взрыв, со...",Мир,В Лондоне при взрыве пострадал рабочий В центр...
4,Польша начнет массовый снос советских памятников,Власти Польши планируют снести почти 30 памятн...,Мир,Польша начнет массовый снос советских памятник...


In [20]:
df_final['cleaned_text'] = df_final['content'].apply(preprocess_text)

In [21]:
X = df_final['cleaned_text']
y = df_final['topic']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=RANDOM_STATE,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

In [22]:
X

,cleaned_text
0,государствен департамент сша устраива бумажн п...
1,южн осет прос росс отмен экономическ санкц вла...
2,выходн москв прошл пушкинск велоноч сентябр со...
3,лондон взрыв пострада рабоч центр лондон февра...
4,польш начнет массов снос советск памятник влас...
...,...
99995,кгб белорусс обвин джозеф ке коммерческ шпиона...
99996,летн бабушк казахста покор интернет рэп житух ...
99997,медвед подготов закон увольнен связ утрат дове...
99998,собак науч наход детск порнограф запах правоох...


In [23]:
y

,topic
0,Мир
1,Бывший СССР
2,Дом
3,Мир
4,Мир
...,...
99995,Бывший СССР
99996,Интернет и СМИ
99997,Россия
99998,Интернет и СМИ


# Dummy

In [24]:
dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_val)
print("бейзлайн\n", classification_report(y_val, y_pred_dummy, zero_division=0))

бейзлайн
                    precision    recall  f1-score   support

                        0.00      0.00      0.00         6
   69-я параллель       0.00      0.00      0.00        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.00      0.01      0.00       200
      Бывший СССР       0.07      0.07      0.07      1444
              Дом       0.03      0.03      0.03       588
         Из жизни       0.02      0.02      0.02       747
   Интернет и СМИ       0.06      0.06      0.06      1208
             Крым       0.00      0.00      0.00        18
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.07      0.08      0.08      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.18      0.18      0.18      3698
  Наука и техника       0.08      0.09      0.08      1437
      Путешествия       0.02      0.02      0.02       174
           Россия       0.22      0.22      0

# CountVectorizer

In [25]:
CV = Pipeline([
    ('vect', CountVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter= 500))
])

CV.fit(X_train, y_train)
y_pred_cv = CV.predict(X_val)
print("Countvector \n", classification_report(y_val, y_pred_cv, zero_division=0))

Countvector 
                    precision    recall  f1-score   support

                        0.00      0.00      0.00         6
   69-я параллель       0.60      0.43      0.50        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.49      0.51      0.50       200
      Бывший СССР       0.81      0.82      0.81      1444
              Дом       0.81      0.84      0.82       588
         Из жизни       0.57      0.62      0.59       747
   Интернет и СМИ       0.72      0.70      0.71      1208
             Крым       0.33      0.28      0.30        18
    Культпросвет        0.20      0.22      0.21         9
         Культура       0.87      0.87      0.87      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.80      0.79      0.79      3698
  Наука и техника       0.79      0.81      0.80      1437
      Путешествия       0.69      0.68      0.69       174
           Россия       0.79      0.78   

# TfidfVectorizer

In [26]:
TD_CV = Pipeline([
    ('vect', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000))
])

TD_CV.fit(X_train, y_train)
y_pred_td_cv = TD_CV.predict(X_val)
print("tfidf\n", classification_report(y_val, y_pred_td_cv, zero_division=0))

tfidf
                    precision    recall  f1-score   support

                        0.00      0.00      0.00         6
   69-я параллель       0.57      0.60      0.58        35
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.35      0.69      0.47       200
      Бывший СССР       0.78      0.86      0.82      1444
              Дом       0.74      0.88      0.80       588
         Из жизни       0.49      0.76      0.59       747
   Интернет и СМИ       0.74      0.73      0.74      1208
             Крым       0.15      0.44      0.23        18
    Культпросвет        0.19      0.33      0.24         9
         Культура       0.86      0.88      0.87      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.84      0.76      0.80      3698
  Наука и техника       0.80      0.81      0.81      1437
      Путешествия       0.50      0.79      0.61       174
           Россия       0.86      0.70      0.77

# tfidf показал себя буквально чуть чуть лучше чем CV

# сделаем тюнинг параметров

пытался сделать такой перебор:

param_grid = {
    # диапазон n-грамм
    'vect__ngram_range': [(1, 1), (1, 2), (1, 3)],
    
    # Минимальная частота слова
    'vect__min_df': [5, 10, 20],
    
    # Максимальная частота слова
    'vect__max_df': [0.7, 0.8, 0.9],
    
    # Использование sublinear_tf
    'vect__sublinear_tf': [True],
    
    # Параметр регуляризации C
    'clf__C': [0.01, 0.1, 1.0, 10.0],
    
    # Тип регуляризации, можно попробовать l1,но использую l2
    'clf__penalty': ['l2']
}

но уже жду 32 минуты. сократил перебор

не успел параметры перебрать... все равно

In [31]:

param_grid = {
    'vect__ngram_range': [(1, 1), (1, 2)],
    'vect__min_df': [20, 40],
    'vect__max_df': [0.6, 0.8],
    'vect__sublinear_tf': [True],
    'clf__C': [0.1, 1, 10]
}

grid = GridSearchCV(
    estimator=TD_CV,
    param_grid=param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=3,
    verbose=2
)

grid.fit(X_train, y_train)

print("best params", grid.best_params_)
print("Лучший score", grid.best_score_)
best_model = grid.best_estimator_

Fitting 3 folds for each of 24 candidates, totalling 72 fits
best params {'clf__C': 10, 'vect__max_df': 0.8, 'vect__min_df': 20, 'vect__ngram_range': (1, 2), 'vect__sublinear_tf': True}
Лучший score 0.8135387416222665


In [33]:
y_pred = best_model.predict(X_test)
print("finality \n", classification_report(y_test, y_pred, zero_division=0))

finality 
                    precision    recall  f1-score   support

                        1.00      0.20      0.33         5
   69-я параллель       0.84      0.62      0.71        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.54      0.54      0.54       200
      Бывший СССР       0.81      0.87      0.84      1445
              Дом       0.80      0.85      0.83       588
         Из жизни       0.61      0.69      0.65       747
   Интернет и СМИ       0.74      0.76      0.75      1209
             Крым       0.45      0.50      0.47        18
    Культпросвет        0.20      0.22      0.21         9
         Культура       0.86      0.88      0.87      1455
          Легпром       0.00      0.00      0.00         3
              Мир       0.83      0.80      0.82      3697
  Наука и техника       0.84      0.85      0.84      1438
      Путешествия       0.67      0.81      0.73       173
           Россия       0.85      0.78      